In [1]:
from __future__ import annotations

import importlib.metadata
import json
import os
import shlex
import subprocess
import sys
from pathlib import Path

PYTORCH_INDEX = "https://download.pytorch.org/whl/cu118"
BOOTSTRAP_REPORT = Path("/kaggle/working/nexar_v81_bootstrap.json")


def run(command: list[str], label: str) -> subprocess.CompletedProcess[str]:
    print(f"[{label}]", " ".join(shlex.quote(part) for part in command))
    result = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout[-16000:])
    return result


def gpu_name() -> str:
    result = run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        "GPU discovery",
    )
    return result.stdout.strip().splitlines()[0] if result.returncode == 0 else "CPU/unknown"


PROBE_SOURCE = r"""
import json
import ctypes
import torch

payload = {
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "architectures": torch.cuda.get_arch_list() if torch.cuda.is_available() else [],
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
if torch.cuda.is_available():
    x = torch.randn(256, 256, device="cuda", dtype=torch.float16)
    y = x @ x
    torch.cuda.synchronize()
    payload["finite"] = bool(torch.isfinite(y).all().item())
print(json.dumps(payload))
"""


def torch_probe() -> tuple[bool, str]:
    result = subprocess.run(
        [sys.executable, "-c", PROBE_SOURCE],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    return result.returncode == 0, result.stdout


name = gpu_name()
is_p100 = "P100" in name.upper()
probe_ok, probe_output = torch_probe()
print("Initial PyTorch probe:", "PASS" if probe_ok else "FAIL")
print(probe_output[-12000:])

needs_p100_wheel = is_p100 and (
    not probe_ok or '"sm_60"' not in probe_output
)

if needs_p100_wheel:
    # Remove optional binary packages tied to the newer Kaggle Torch build.
    run(
        [
            sys.executable, "-m", "pip", "uninstall", "-y",
            "torchvision", "torchaudio", "torchcodec", "torchao", "xformers",
        ],
        "remove incompatible optional Torch binaries",
    )

    # IMPORTANT: no --no-deps. The cu118 runtime packages must be installed too.
    install = run(
        [
            sys.executable, "-m", "pip", "install",
            "--no-cache-dir", "--upgrade", "--force-reinstall",
            "torch==2.5.1",
            "--index-url", PYTORCH_INDEX,
        ],
        "install P100-compatible PyTorch and CUDA 11.8 dependencies",
    )
    if install.returncode != 0:
        raise RuntimeError(
            "P100 PyTorch installation failed. Review the complete pip output above."
        )

    probe_ok, probe_output = torch_probe()
    print("Post-install PyTorch probe:", "PASS" if probe_ok else "FAIL")
    print(probe_output[-12000:])

if not probe_ok:
    raise RuntimeError(
        "PyTorch CUDA probe failed. Start a completely fresh Kaggle GPU session "
        "and run this notebook from the first cell."
    )
if is_p100 and '"sm_60"' not in probe_output:
    raise RuntimeError("Installed PyTorch still does not contain sm_60 kernels.")

BOOTSTRAP_REPORT.write_text(
    json.dumps(
        {
            "gpu": name,
            "p100": is_p100,
            "probe": probe_output,
        },
        indent=2,
    ),
    encoding="utf-8",
)
print("Bootstrap complete:", BOOTSTRAP_REPORT)


[GPU discovery] nvidia-smi --query-gpu=name --format=csv,noheader
Tesla P100-PCIE-16GB

Initial PyTorch probe: FAIL
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check th

## 1. نصب فقط وابستگی‌های سبک و Missing


In [2]:
from __future__ import annotations

import importlib.util
import subprocess
import sys

REQUIRED = {
    "cv2": "opencv-python-headless>=4.10,<5",
    "datasets": "datasets>=3.2,<5",
    "huggingface_hub": "huggingface_hub>=0.27,<2",
    "pandas": "pandas>=2.2,<3",
    "tqdm": "tqdm>=4.66,<5",
}

missing = [
    package for module, package in REQUIRED.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    command = [
        sys.executable, "-m", "pip", "install",
        "--no-cache-dir", "--upgrade", *missing,
    ]
    print("Installing:", missing)
    subprocess.check_call(command)
else:
    print("All lightweight dependencies are already available.")


All lightweight dependencies are already available.


## 2. Imports، Config متمرکز و مسیرهای Kaggle


In [3]:
from __future__ import annotations

import contextlib
import gc
import hashlib
import io
import json
import math
import os
import random
import re
import shutil
import tempfile
import time
import traceback
import urllib.parse
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterable, Iterator, Mapping, Sequence

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import datasets as hf_datasets
import fsspec
import requests
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from IPython.display import display
from tqdm.auto import tqdm

try:
    import matplotlib.pyplot as plt
except Exception as error:
    plt = None
    print("Matplotlib unavailable; numerical pipeline will continue:", repr(error))


@dataclass(frozen=True)
class Config:
    stage: str = os.getenv("NEXAR_STAGE", "reproduce").strip().lower()
    seed: int = 42

    dataset_repo: str = "nexar-ai/nexar_collision_prediction"
    checkpoint_repo: str = "zhiyaowang/VideoMaev2-giant-nexar-solution"
    checkpoint_name: str = "best.pth"
    checkpoint_sha256: str = (
        "dbf68c391c3322fe23a502a5e80afb7b3184221c63d62b25983507435fb39af7"
    )

    train_split: str = "train"
    test_split: str = "test"

    image_size: int = 224
    resize_short_side: int = 224
    num_frames: int = 16
    frame_interval: int = 1
    training_window_stride: int = 2
    positive_horizon_seconds: float = 1.5
    temperature: float = 2.0

    # Exact released-checkpoint inference uses the final visible window.
    temporal_tta_offsets: tuple[int, ...] = (0,)
    horizontal_flip_tta: bool = False

    # P100-safe defaults.
    inference_batch_size: int = 1
    feature_batch_size: int = 1
    amp: bool = True

    # Research head fine-tuning budget.
    positive_windows_per_video: int = 6
    negative_windows_per_video: int = 6
    folds: int = 5
    head_epochs: int = 30
    head_lr: float = 3e-4
    head_weight_decay: float = 1e-3
    head_hidden_dim: int = 256
    head_patience: int = 6

    resume: bool = True
    test_retries: int = 3
    systemic_failure_limit: int = 3
    remote_timeout_seconds: int = 180
    evaluate_official_solution: bool = True
    verify_checkpoint_hash: bool = True

    smoke_train_videos: int = 8
    smoke_test_videos: int = 8

    @property
    def max_train_videos(self) -> int | None:
        return self.smoke_train_videos if self.stage == "smoke" else None

    @property
    def max_test_videos(self) -> int | None:
        return self.smoke_test_videos if self.stage == "smoke" else None


CFG = Config()
VALID_STAGES = {"smoke", "reproduce", "cache_features", "train_head"}
if CFG.stage not in VALID_STAGES:
    raise ValueError(f"NEXAR_STAGE must be one of {sorted(VALID_STAGES)}, got {CFG.stage!r}")

KAGGLE_INPUT_DIR = Path("/kaggle/input")
KAGGLE_WORKING_DIR = Path("/kaggle/working")
OUTPUT_DIR = KAGGLE_WORKING_DIR / "outputs" / f"videomaev2_giant_v82_{CFG.stage}"
CACHE_DIR = KAGGLE_WORKING_DIR / "nexar_v82_cache"
FEATURE_DIR = CACHE_DIR / "train_features"
PREDICTION_DIR = OUTPUT_DIR / "predictions"
FIGURE_DIR = OUTPUT_DIR / "figures"

for directory in (OUTPUT_DIR, CACHE_DIR, FEATURE_DIR, PREDICTION_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("A Kaggle GPU accelerator is required for VideoMAEv2-Giant.")

AMP_ENABLED = CFG.amp and DEVICE.type == "cuda"
AMP_DTYPE = torch.float16
print(json.dumps({
    "config": asdict(CFG),
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "device": torch.cuda.get_device_name(0),
    "architectures": torch.cuda.get_arch_list(),
    "output_dir": str(OUTPUT_DIR),
}, indent=2, default=str))


{
  "config": {
    "stage": "reproduce",
    "seed": 42,
    "dataset_repo": "nexar-ai/nexar_collision_prediction",
    "checkpoint_repo": "zhiyaowang/VideoMaev2-giant-nexar-solution",
    "checkpoint_name": "best.pth",
    "checkpoint_sha256": "dbf68c391c3322fe23a502a5e80afb7b3184221c63d62b25983507435fb39af7",
    "train_split": "train",
    "test_split": "test",
    "image_size": 224,
    "resize_short_side": 224,
    "num_frames": 16,
    "frame_interval": 1,
    "training_window_stride": 2,
    "positive_horizon_seconds": 1.5,
    "temperature": 2.0,
    "temporal_tta_offsets": [
      0
    ],
    "horizontal_flip_tta": false,
    "inference_batch_size": 1,
    "feature_batch_size": 1,
    "amp": true,
    "positive_windows_per_video": 6,
    "negative_windows_per_video": 6,
    "folds": 5,
    "head_epochs": 30,
    "head_lr": 0.0003,
    "head_weight_decay": 0.001,
    "head_hidden_dim": 256,
    "head_patience": 6,
    "resume": true,
    "test_retries": 3,
    "systemic_failu

## 3. Reproducibility، Atomic I/O، Resume و Metricهای رسمی


In [4]:
def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Decoder/input shapes are stable; benchmark improves P100 throughput.
    torch.backends.cudnn.benchmark = True


def atomic_json(payload: Any, path: Path) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )
    os.replace(temporary, path)


def append_jsonl(payload: Mapping[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    line = json.dumps(dict(payload), ensure_ascii=False, default=str)
    with path.open("a", encoding="utf-8") as stream:
        stream.write(line + "\n")


def read_jsonl_recover(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    rows: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as stream:
        for number, line in enumerate(stream, start=1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                print(f"Ignoring incomplete JSONL tail at {path}:{number}")
                break
    return pd.DataFrame(rows)


def completed_ids(path: Path) -> set[str]:
    frame = read_jsonl_recover(path)
    if frame.empty or "status" not in frame:
        return set()
    latest = frame.drop_duplicates("video_id", keep="last")
    return set(latest.loc[latest.status.eq("ok"), "video_id"].astype(str))


def average_precision_numpy(y_true: Sequence[int], y_score: Sequence[float]) -> float:
    """Equivalent to sklearn Average Precision, including tied-score handling."""
    y = np.asarray(y_true, dtype=np.int64)
    score = np.asarray(y_score, dtype=np.float64)
    if y.ndim != 1 or score.ndim != 1 or len(y) != len(score):
        raise ValueError("y_true and y_score must be same-length 1D arrays.")
    positives = int(y.sum())
    if positives == 0:
        return float("nan")

    order = np.argsort(-score, kind="mergesort")
    y = y[order]
    score = score[order]
    tp = np.cumsum(y)
    fp = np.cumsum(1 - y)

    distinct = np.where(np.diff(score))[0]
    threshold_indices = np.r_[distinct, len(score) - 1]
    precision = tp[threshold_indices] / (tp[threshold_indices] + fp[threshold_indices])
    recall = tp[threshold_indices] / positives
    previous_recall = np.r_[0.0, recall[:-1]]
    return float(np.sum((recall - previous_recall) * precision))


def binary_metrics(
    y_true: Sequence[int],
    scores: Sequence[float],
    threshold: float = 0.5,
) -> dict[str, float]:
    y = np.asarray(y_true, dtype=np.int64)
    score = np.asarray(scores, dtype=np.float64)
    pred = (score >= threshold).astype(np.int64)
    tp = int(((pred == 1) & (y == 1)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    return {
        "accuracy": float((tp + tn) / max(len(y), 1)),
        "balanced_accuracy": float((recall + specificity) / 2),
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "f1": float(2 * precision * recall / max(precision + recall, 1e-12)),
        "average_precision": average_precision_numpy(y, score),
        "threshold": float(threshold),
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
    }


def select_accuracy_threshold(
    y_true: Sequence[int],
    scores: Sequence[float],
) -> tuple[float, dict[str, float]]:
    """Select threshold on validation/OOF only; never call on Test for tuning."""
    y = np.asarray(y_true, dtype=np.int64)
    score = np.asarray(scores, dtype=np.float64)
    candidates = np.unique(np.r_[0.0, score, 1.0])
    best_threshold = 0.5
    best_metrics = binary_metrics(y, score, best_threshold)
    for threshold in candidates:
        metrics = binary_metrics(y, score, float(threshold))
        key = (
            metrics["accuracy"],
            metrics["balanced_accuracy"],
            metrics["f1"],
            -abs(float(threshold) - 0.5),
        )
        best_key = (
            best_metrics["accuracy"],
            best_metrics["balanced_accuracy"],
            best_metrics["f1"],
            -abs(best_threshold - 0.5),
        )
        if key > best_key:
            best_threshold = float(threshold)
            best_metrics = metrics
    return best_threshold, best_metrics


seed_everything(CFG.seed)
assert np.isclose(
    average_precision_numpy([1, 0, 1, 0], [0.5, 0.5, 0.2, 0.2]),
    0.5,
)
print("Metric and resume utilities passed.")


Metric and resume utilities passed.


## 4. دریافت Artifactهای رسمی و Checkpoint 12.1GB


In [5]:
def resolve_hf_token() -> str | None:
    token = os.getenv("HF_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None


def discover_attached_file(filename: str) -> Path | None:
    if not KAGGLE_INPUT_DIR.exists():
        return None
    candidates = list(KAGGLE_INPUT_DIR.rglob(filename))
    if not candidates:
        return None
    candidates.sort(key=lambda item: (
        "nexar" not in str(item).lower(),
        "videomae" not in str(item).lower(),
        len(str(item)),
    ))
    return candidates[0]


def sha256_file(path: Path, chunk_size: int = 32 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def ensure_hf_file(
    repo_id: str,
    filename: str,
    local_dir: Path,
    *,
    repo_type: str | None = None,
) -> Path:
    attached = discover_attached_file(filename)
    if attached is not None:
        print("Using attached Kaggle input:", attached)
        return attached

    local_dir.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {repo_id}/{filename}. Internet must be enabled.")
    downloaded = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        repo_type=repo_type,
        local_dir=str(local_dir),
        token=resolve_hf_token(),
    )
    return Path(downloaded)


# Small official evaluator files.
ARTIFACTS: dict[str, Path] = {}
for filename in (
    "sample_submission.csv",
    "solution.csv",
    "time_to_accident_test_map.csv",
    "evaluate_submission.py",
):
    try:
        ARTIFACTS[filename] = ensure_hf_file(
            CFG.dataset_repo,
            filename,
            CACHE_DIR / "official",
            repo_type="dataset",
        )
    except Exception as error:
        if filename in {"solution.csv", "time_to_accident_test_map.csv", "evaluate_submission.py"}:
            print(f"Optional official file unavailable ({filename}):", repr(error))
        else:
            raise

attached_checkpoint = discover_attached_file(CFG.checkpoint_name)
if attached_checkpoint is None:
    free_gib = shutil.disk_usage(KAGGLE_WORKING_DIR).free / 2**30
    if free_gib < 14.0:
        raise RuntimeError(
            f"Only {free_gib:.1f} GiB is free in /kaggle/working. "
            "Attach the released 12.1-GB best.pth as a Kaggle Dataset input "
            "instead of downloading it into the session."
        )

CHECKPOINT = ensure_hf_file(
    CFG.checkpoint_repo,
    CFG.checkpoint_name,
    CACHE_DIR / "checkpoint",
)

size_gib = CHECKPOINT.stat().st_size / 2**30
print(f"Checkpoint: {CHECKPOINT} ({size_gib:.2f} GiB)")
if size_gib < 10.0:
    raise RuntimeError(
        "The discovered best.pth is too small and is probably not the released "
        "VideoMAEv2-Giant checkpoint."
    )
if CFG.verify_checkpoint_hash:
    digest = sha256_file(CHECKPOINT)
    if digest != CFG.checkpoint_sha256:
        raise RuntimeError(
            f"Checkpoint SHA256 mismatch: expected {CFG.checkpoint_sha256}, got {digest}"
        )
    print("Checkpoint SHA256 verified.")


sample_submission.csv: 0.00B [00:00, ?B/s]

solution.csv: 0.00B [00:00, ?B/s]

time_to_accident_test_map.csv: 0.00B [00:00, ?B/s]

evaluate_submission.py: 0.00B [00:00, ?B/s]

best.pth:   0%|          | 0.00/12.1G [00:00<?, ?B/s]

Checkpoint: /kaggle/working/nexar_v82_cache/checkpoint/best.pth (11.31 GiB)
Checkpoint SHA256 verified.


## 5. Raw Hugging Face Streaming Loader بدون TorchCodec


In [6]:
def disable_hf_video_backends() -> dict[str, Any]:
    previous: dict[str, Any] = {}
    config = hf_datasets.config
    for name in (
        "TORCHCODEC_AVAILABLE",
        "TORCHVISION_AVAILABLE",
        "DECORD_AVAILABLE",
        "PYAV_AVAILABLE",
    ):
        if hasattr(config, name):
            previous[name] = getattr(config, name)
            setattr(config, name, False)
    return previous


class RawStreamingRecords:
    """Bypass Video decoding while preserving the original HF record key."""

    def __init__(self, dataset: Any, split: str) -> None:
        self.dataset = dataset
        self.split = split

    def __iter__(self) -> Iterator[dict[str, Any]]:
        ex_iterable = getattr(self.dataset, "_ex_iterable", None)
        if ex_iterable is None:
            raise RuntimeError(
                "This datasets version exposes no raw _ex_iterable. "
                "The formatted Video feature is intentionally bypassed because it may "
                "activate an ABI-incompatible TorchCodec decoder."
            )
        for key, example in ex_iterable:
            record = dict(example)
            record["__hf_key__"] = str(key)
            yield record


def load_raw_split(split: str) -> RawStreamingRecords:
    disabled = disable_hf_video_backends()
    print("Disabled HF video backends:", disabled)
    dataset = load_dataset(
        CFG.dataset_repo,
        split=split,
        streaming=True,
        token=resolve_hf_token(),
    )
    return RawStreamingRecords(dataset, split)


def canonical_video_id(value: Any) -> str | None:
    if value is None:
        return None
    raw = str(value).strip()
    if not raw:
        return None
    clean = raw.split("?", 1)[0].split("#", 1)[0].rstrip("/")
    stem = Path(clean).stem.strip()
    if stem.isdigit():
        return f"{int(stem):05d}"
    matches = re.findall(r"(?<!\d)(\d{5})(?!\d)", stem)
    if matches:
        return matches[-1]
    return stem or None


def video_path_hint(video_field: Any) -> str | None:
    if isinstance(video_field, Mapping):
        value = video_field.get("path")
        return str(value) if value is not None else None
    if isinstance(video_field, (str, os.PathLike)):
        return str(video_field)
    for name in ("path", "name", "filename"):
        value = getattr(video_field, name, None)
        if value:
            return str(value)
    return None


def example_id(example: Mapping[str, Any], index: int) -> str:
    candidates = [
        example.get("id"),
        example.get("video_id"),
        example.get("clip_id"),
        example.get("uid"),
        example.get("filename"),
        video_path_hint(example.get("video")),
        example.get("__hf_key__"),
    ]
    for candidate in candidates:
        resolved = canonical_video_id(candidate)
        if resolved is not None:
            return resolved
    raise RuntimeError(
        f"Could not resolve true video ID at dataset index {index}; "
        f"keys={sorted(example.keys())}"
    )


VIDEO_SUFFIXES = {".mp4", ".mov", ".m4v", ".avi", ".webm", ".mkv"}


def discover_local_test_videos() -> dict[str, Path]:
    """Index attached Kaggle Test videos by official five-digit ID.

    Local Kaggle inputs are preferred because they avoid remote streaming and are
    reproducible when Internet is disabled. Paths containing 'test' are preferred;
    ambiguous duplicate IDs are intentionally excluded.
    """
    candidates: dict[str, list[Path]] = {}
    if not KAGGLE_INPUT_DIR.exists():
        return {}
    for path in KAGGLE_INPUT_DIR.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in VIDEO_SUFFIXES:
            continue
        identifier = canonical_video_id(path.name)
        if identifier is not None:
            candidates.setdefault(identifier, []).append(path)

    index: dict[str, Path] = {}
    for identifier, paths in candidates.items():
        test_paths = [p for p in paths if "test" in str(p).lower()]
        selected = test_paths if test_paths else paths
        if len(selected) == 1:
            index[identifier] = selected[0]
    print(
        "Local video index:", len(index),
        "unique IDs from", sum(len(v) for v in candidates.values()), "video files",
    )
    return index


LOCAL_TEST_VIDEO_INDEX = discover_local_test_videos()


def _authorization_headers() -> dict[str, str]:
    token = resolve_hf_token()
    return {"Authorization": f"Bearer {token}"} if token else {}


def _copy_binary_stream(source: Any, destination: Path) -> None:
    with destination.open("wb") as output:
        shutil.copyfileobj(source, output, length=8 * 1024 * 1024)


def _http_to_file(url: str, destination: Path) -> None:
    with requests.get(
        url,
        stream=True,
        timeout=(30, CFG.remote_timeout_seconds),
        headers=_authorization_headers(),
    ) as response:
        response.raise_for_status()
        with destination.open("wb") as output:
            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    output.write(chunk)


def _hf_relative_uri(source: str) -> str:
    clean = source.replace("\\", "/").lstrip("/")
    return f"hf://datasets/{CFG.dataset_repo}/{clean}"


def _copy_path_or_uri(source: str | os.PathLike[str], destination: Path) -> str:
    """Copy a local path, HTTP URL, fsspec URI, or HF-relative repo path."""
    text = str(source).strip()
    if not text:
        raise ValueError("Empty video source path.")

    local = Path(text).expanduser()
    if local.exists() and local.is_file():
        with local.open("rb") as stream:
            _copy_binary_stream(stream, destination)
        return "local"

    parsed = urllib.parse.urlparse(text)
    if parsed.scheme in {"http", "https"}:
        _http_to_file(text, destination)
        return parsed.scheme

    attempts = [text]
    if not parsed.scheme:
        attempts.append(_hf_relative_uri(text))

    errors: list[str] = []
    for uri in attempts:
        try:
            storage_options: dict[str, Any] = {}
            token = resolve_hf_token()
            if uri.startswith("hf://") and token:
                storage_options["token"] = token
            with fsspec.open(uri, mode="rb", **storage_options).open() as stream:
                _copy_binary_stream(stream, destination)
            return "fsspec"
        except Exception as error:
            errors.append(f"{uri!r}: {type(error).__name__}: {error}")

    raise FileNotFoundError(
        "Could not open video source as local path, URL, fsspec URI, or "
        f"Hugging Face relative path. Attempts: {errors}"
    )


@contextlib.contextmanager
def temporary_video(
    video_field: Any,
    *,
    identifier: str | None = None,
    hf_key: str | None = None,
) -> Iterator[Path]:
    """Materialize one video robustly and delete it immediately afterwards."""

    if identifier is not None and identifier in LOCAL_TEST_VIDEO_INDEX:
        # Attached Kaggle data: no copy and no network traffic.
        yield LOCAL_TEST_VIDEO_INDEX[identifier]
        return

    source_hint = video_path_hint(video_field)
    suffix = Path(source_hint).suffix.lower() if source_hint else ".mp4"
    if suffix not in VIDEO_SUFFIXES:
        suffix = ".mp4"

    handle = tempfile.NamedTemporaryFile(
        prefix="nexar_v82_",
        suffix=suffix,
        delete=False,
    )
    path = Path(handle.name)
    handle.close()

    source_mode = "unknown"
    try:
        if isinstance(video_field, Mapping):
            raw = video_field.get("bytes")
            source = video_field.get("path")
            if raw is not None:
                path.write_bytes(bytes(raw))
                source_mode = "mapping-bytes"
            elif source is not None:
                source_mode = _copy_path_or_uri(source, path)
            elif hf_key:
                source_mode = _copy_path_or_uri(hf_key, path)
            else:
                raise TypeError(
                    f"Unsupported raw video mapping keys: {sorted(video_field.keys())}"
                )
        elif isinstance(video_field, (bytes, bytearray, memoryview)):
            path.write_bytes(bytes(video_field))
            source_mode = "bytes"
        elif isinstance(video_field, (str, os.PathLike)):
            source_mode = _copy_path_or_uri(video_field, path)
        elif hasattr(video_field, "read"):
            if hasattr(video_field, "seek"):
                try:
                    video_field.seek(0)
                except Exception:
                    pass
            _copy_binary_stream(video_field, path)
            source_mode = "file-like"
        elif hf_key:
            source_mode = _copy_path_or_uri(hf_key, path)
        else:
            raise TypeError(f"Unsupported video field: {type(video_field)!r}")

        size = path.stat().st_size
        if size < 1024:
            raise IOError(
                f"Materialized video is unexpectedly small: {size} bytes; "
                f"mode={source_mode}, source={source_hint!r}, hf_key={hf_key!r}"
            )
        yield path
    finally:
        path.unlink(missing_ok=True)


# Offline materialization tests: bytes, local path and file-like input.
_test_video_bytes = b"0" * 2048
with temporary_video({"bytes": _test_video_bytes, "path": None}) as _path:
    assert _path.stat().st_size == len(_test_video_bytes)
with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as _src:
    _src.write(_test_video_bytes)
    _src_path = Path(_src.name)
try:
    with temporary_video(str(_src_path)) as _path:
        assert _path.stat().st_size == len(_test_video_bytes)
finally:
    _src_path.unlink(missing_ok=True)
print("Robust raw/local/URI video loader defined and offline tests passed.")


Local video index: 0 unique IDs from 0 video files
Robust raw/local/URI video loader defined and offline tests passed.


## 6. Causal Video Sampling و Preprocessing مطابق VideoMAEv2‑Giant


In [7]:
IMAGENET_MEAN = np.asarray([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.asarray([0.229, 0.224, 0.225], dtype=np.float32)


def probe_video(path: Path) -> dict[str, float | int]:
    capture = cv2.VideoCapture(str(path))
    if not capture.isOpened():
        raise IOError(f"Cannot open video: {path}")
    try:
        fps = float(capture.get(cv2.CAP_PROP_FPS))
        frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
        width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
        if fps <= 0 or frame_count <= 0 or width <= 0 or height <= 0:
            raise IOError(
                f"Invalid video metadata: fps={fps}, frames={frame_count}, "
                f"size={width}x{height}"
            )
        return {
            "fps": fps,
            "frame_count": frame_count,
            "width": width,
            "height": height,
            "duration": frame_count / fps,
        }
    finally:
        capture.release()


def causal_end_frame(cutoff_seconds: float, fps: float, frame_count: int) -> int:
    # Strictly before cutoff. floor(cutoff * fps) - 1 prevents future-frame leakage.
    end = min(int(math.floor(cutoff_seconds * fps)) - 1, frame_count - 1)
    return max(end, 0)


def clip_indices_from_end(
    end_frame: int,
    frame_count: int,
    *,
    offset_frames: int = 0,
) -> np.ndarray:
    end = max(0, min(end_frame - offset_frames, frame_count - 1))
    span = (CFG.num_frames - 1) * CFG.frame_interval
    start = end - span
    indices = start + np.arange(CFG.num_frames, dtype=np.int64) * CFG.frame_interval
    indices = np.clip(indices, 0, frame_count - 1)
    if len(indices) != CFG.num_frames or np.any(indices > end):
        raise AssertionError("Invalid causal clip indices.")
    return indices


def decode_frame_indices(path: Path, indices: np.ndarray) -> np.ndarray:
    capture = cv2.VideoCapture(str(path))
    if not capture.isOpened():
        raise IOError(f"Cannot decode video: {path}")

    requested = np.asarray(indices, dtype=np.int64)
    unique = sorted(set(requested.tolist()))
    decoded: dict[int, np.ndarray] = {}
    try:
        capture.set(cv2.CAP_PROP_POS_FRAMES, unique[0])
        current = unique[0]
        wanted = set(unique)
        while current <= unique[-1]:
            ok = capture.grab()
            if not ok:
                break
            if current in wanted:
                retrieved, frame = capture.retrieve()
                if retrieved and frame is not None:
                    decoded[current] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            current += 1
    finally:
        capture.release()

    if not decoded:
        raise IOError("No requested frames were decoded.")

    available = sorted(decoded)
    frames: list[np.ndarray] = []
    for index in requested:
        key = int(index)
        if key not in decoded:
            key = min(available, key=lambda item: abs(item - key))
        frames.append(decoded[key])
    return np.stack(frames)


def resize_center_crop(frame: np.ndarray) -> np.ndarray:
    height, width = frame.shape[:2]
    scale = CFG.resize_short_side / min(height, width)
    resized_width = max(CFG.image_size, int(round(width * scale)))
    resized_height = max(CFG.image_size, int(round(height * scale)))
    resized = cv2.resize(
        frame,
        (resized_width, resized_height),
        interpolation=cv2.INTER_CUBIC,
    )
    top = (resized_height - CFG.image_size) // 2
    left = (resized_width - CFG.image_size) // 2
    cropped = resized[
        top:top + CFG.image_size,
        left:left + CFG.image_size,
    ]
    if cropped.shape != (CFG.image_size, CFG.image_size, 3):
        raise AssertionError(f"Unexpected crop shape: {cropped.shape}")
    return cropped


def preprocess_clip(frames: np.ndarray, horizontal_flip: bool = False) -> torch.Tensor:
    processed = np.stack([resize_center_crop(frame) for frame in frames])
    if horizontal_flip:
        processed = processed[:, :, ::-1, :].copy()
    values = processed.astype(np.float32) / 255.0
    values = (values - IMAGENET_MEAN) / IMAGENET_STD
    # Official model input: B,C,T,H,W.
    return torch.from_numpy(values).permute(3, 0, 1, 2).contiguous()


def final_visible_clips(path: Path) -> tuple[list[torch.Tensor], dict[str, Any]]:
    metadata = probe_video(path)
    end = metadata["frame_count"] - 1
    clips: list[torch.Tensor] = []
    for offset in CFG.temporal_tta_offsets:
        indices = clip_indices_from_end(
            int(end),
            int(metadata["frame_count"]),
            offset_frames=int(offset),
        )
        frames = decode_frame_indices(path, indices)
        clips.append(preprocess_clip(frames, horizontal_flip=False))
        if CFG.horizontal_flip_tta:
            clips.append(preprocess_clip(frames, horizontal_flip=True))
    return clips, {**metadata, "final_end_frame": int(end)}


# Explicit causality tests.
for cutoff in (0.5, 1.0, 1.5, 10.0):
    fps = 30.0
    count = 1200
    end = causal_end_frame(cutoff, fps, count)
    indices = clip_indices_from_end(end, count)
    assert np.all(indices / fps < cutoff)
print("Causal sampling and preprocessing definitions passed.")


Causal sampling and preprocessing definitions passed.


## 7. معماری Pure‑PyTorch دقیق VideoMAEv2‑Giant بدون timm/torchvision


In [8]:
def drop_path(
    x: torch.Tensor,
    drop_prob: float = 0.0,
    training: bool = False,
) -> torch.Tensor:
    if drop_prob == 0.0 or not training:
        return x
    keep_prob = 1.0 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)
    random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
    random_tensor.floor_()
    return x.div(keep_prob) * random_tensor


class DropPath(nn.Module):
    def __init__(self, probability: float = 0.0) -> None:
        super().__init__()
        self.drop_prob = probability

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return drop_path(x, self.drop_prob, self.training)


class Mlp(nn.Module):
    def __init__(
        self,
        in_features: int,
        hidden_features: int,
        out_features: int,
        drop: float = 0.0,
    ) -> None:
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.drop(self.fc2(self.act(self.fc1(x))))


class Attention(nn.Module):
    def __init__(
        self,
        dim: int,
        num_heads: int,
        qkv_bias: bool = True,
        qk_scale: float | None = None,
        attn_drop: float = 0.0,
        proj_drop: float = 0.0,
    ) -> None:
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = float(qk_scale or head_dim ** -0.5)
        all_head_dim = head_dim * num_heads
        self.qkv = nn.Linear(dim, all_head_dim * 3, bias=False)
        if qkv_bias:
            self.q_bias = nn.Parameter(torch.zeros(all_head_dim))
            self.v_bias = nn.Parameter(torch.zeros(all_head_dim))
        else:
            self.q_bias = None
            self.v_bias = None
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(all_head_dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, tokens, _ = x.shape
        qkv_bias = None
        if self.q_bias is not None:
            qkv_bias = torch.cat(
                (
                    self.q_bias,
                    torch.zeros_like(self.v_bias, requires_grad=False),
                    self.v_bias,
                )
            )
        qkv = F.linear(x, self.qkv.weight, qkv_bias)
        qkv = qkv.reshape(batch, tokens, 3, self.num_heads, -1)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        query, key, value = qkv.unbind(0)

        # PyTorch SDPA is mathematically equivalent and avoids materializing
        # the full attention matrix when an efficient backend is available.
        try:
            output = F.scaled_dot_product_attention(
                query,
                key,
                value,
                dropout_p=self.attn_drop.p if self.training else 0.0,
                scale=self.scale,
            )
        except (TypeError, RuntimeError) as error:
            if "out of memory" in str(error).lower():
                raise
            attention = (query * self.scale) @ key.transpose(-2, -1)
            attention = self.attn_drop(attention.softmax(dim=-1))
            output = attention @ value

        output = output.transpose(1, 2).reshape(batch, tokens, -1)
        return self.proj_drop(self.proj(output))


class Block(nn.Module):
    def __init__(
        self,
        dim: int,
        num_heads: int,
        mlp_ratio: float,
        qkv_bias: bool,
        drop: float,
        attn_drop: float,
        drop_path_probability: float,
        eps: float,
    ) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=eps)
        self.attn = Attention(
            dim,
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            attn_drop=attn_drop,
            proj_drop=drop,
        )
        self.drop_path = (
            DropPath(drop_path_probability)
            if drop_path_probability > 0.0
            else nn.Identity()
        )
        self.norm2 = nn.LayerNorm(dim, eps=eps)
        hidden = int(dim * mlp_ratio)
        self.mlp = Mlp(dim, hidden, dim, drop=drop)
        self.gamma_1 = None
        self.gamma_2 = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class PatchEmbed(nn.Module):
    def __init__(
        self,
        img_size: int,
        patch_size: int,
        in_channels: int,
        embed_dim: int,
        num_frames: int,
        tubelet_size: int,
    ) -> None:
        super().__init__()
        self.img_size = (img_size, img_size)
        self.patch_size = (patch_size, patch_size)
        self.tubelet_size = tubelet_size
        spatial = (img_size // patch_size) ** 2
        self.num_patches = spatial * (num_frames // tubelet_size)
        self.proj = nn.Conv3d(
            in_channels,
            embed_dim,
            kernel_size=(tubelet_size, patch_size, patch_size),
            stride=(tubelet_size, patch_size, patch_size),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, _, _, height, width = x.shape
        if (height, width) != self.img_size:
            raise ValueError(
                f"Input size {(height, width)} does not match {self.img_size}"
            )
        return self.proj(x).flatten(2).transpose(1, 2)


def sinusoid_encoding(num_positions: int, dimension: int) -> torch.Tensor:
    positions = np.arange(num_positions, dtype=np.float64)[:, None]
    dimensions = np.arange(dimension, dtype=np.float64)[None, :]
    angles = positions / np.power(10000.0, 2 * (dimensions // 2) / dimension)
    table = np.empty((num_positions, dimension), dtype=np.float32)
    table[:, 0::2] = np.sin(angles[:, 0::2])
    table[:, 1::2] = np.cos(angles[:, 1::2])
    return torch.from_numpy(table).unsqueeze(0)


class VideoMAEv2Giant(nn.Module):
    """State-dict-compatible implementation of OpenGVLab VideoMAEv2-Giant."""

    def __init__(
        self,
        num_classes: int = 2,
        *,
        use_mean_pooling: bool = False,
        initialize: bool = False,
    ) -> None:
        super().__init__()
        embed_dim = 1408
        depth = 40
        num_heads = 16
        mlp_ratio = 48 / 11
        eps = 1e-6

        self.num_classes = num_classes
        self.num_features = self.embed_dim = embed_dim
        self.patch_embed = PatchEmbed(
            img_size=224,
            patch_size=14,
            in_channels=3,
            embed_dim=embed_dim,
            num_frames=16,
            tubelet_size=2,
        )
        num_patches = self.patch_embed.num_patches
        self.register_buffer(
            "pos_embed",
            sinusoid_encoding(num_patches, embed_dim),
            persistent=False,
        )
        self.pos_drop = nn.Dropout(0.0)
        self.blocks = nn.ModuleList([
            Block(
                dim=embed_dim,
                num_heads=num_heads,
                mlp_ratio=mlp_ratio,
                qkv_bias=True,
                drop=0.0,
                attn_drop=0.0,
                drop_path_probability=0.0,
                eps=eps,
            )
            for _ in range(depth)
        ])
        # The published base config uses token pooling, but competition
        # checkpoints may be exported from the official fine-tuning script with
        # mean pooling. The checkpoint loader evaluates both variants.
        self.use_mean_pooling = bool(use_mean_pooling)
        self.norm = (
            nn.Identity()
            if self.use_mean_pooling
            else nn.LayerNorm(embed_dim, eps=eps)
        )
        self.fc_norm = (
            nn.LayerNorm(embed_dim, eps=eps)
            if self.use_mean_pooling
            else None
        )
        self.head_dropout = nn.Dropout(0.0)
        self.head = nn.Linear(embed_dim, num_classes)

        if initialize:
            self.apply(self._initialize_weights)

    @staticmethod
    def _initialize_weights(module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.trunc_normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.zeros_(module.bias)
            nn.init.ones_(module.weight)

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        batch = x.shape[0]
        x = self.patch_embed(x)
        x = x + self.pos_embed.expand(batch, -1, -1).to(
            device=x.device,
            dtype=x.dtype,
        )
        x = self.pos_drop(x)
        for block in self.blocks:
            x = block(x)
        if self.fc_norm is not None:
            return self.fc_norm(x.mean(dim=1))
        return self.norm(x[:, 0])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.head_dropout(self.forward_features(x)))


# Build on meta only to validate architecture without allocating ~4 GiB.
with torch.device("meta"):
    META_MODEL = VideoMAEv2Giant(num_classes=2, initialize=False)

PARAMETER_COUNT = sum(parameter.numel() for parameter in META_MODEL.parameters())
print("VideoMAEv2-Giant parameters:", f"{PARAMETER_COUNT:,}")
if not 1_000_000_000 <= PARAMETER_COUNT <= 1_080_000_000:
    raise RuntimeError(f"Unexpected giant parameter count: {PARAMETER_COUNT:,}")
del META_MODEL


VideoMAEv2-Giant parameters: 1,011,613,570


## 8. بارگذاری Strict و Memory‑Aware از best.pth


In [9]:
def iter_tensor_mappings(
    payload: Any,
    path: str = "<root>",
    seen: set[int] | None = None,
) -> Iterator[tuple[str, dict[str, torch.Tensor]]]:
    """Yield every tensor mapping inside a checkpoint without guessing its schema."""

    if seen is None:
        seen = set()
    identity = id(payload)
    if identity in seen:
        return
    seen.add(identity)

    if isinstance(payload, nn.Module):
        state = {
            str(key): value
            for key, value in payload.state_dict().items()
            if isinstance(value, torch.Tensor)
        }
        if state:
            yield f"{path}.state_dict()", state
        return

    if not isinstance(payload, Mapping):
        return

    direct = {
        str(key): value
        for key, value in payload.items()
        if isinstance(value, torch.Tensor)
    }
    if len(direct) >= 2:
        yield path, direct

    for key, value in payload.items():
        if isinstance(value, (Mapping, nn.Module)):
            yield from iter_tensor_mappings(
                value,
                f"{path}.{key}",
                seen,
            )


def key_variants(raw_key: str) -> list[str]:
    """Generate wrapper-free and classifier-compatible key candidates."""

    clean = str(raw_key).strip().replace("._orig_mod.", ".")
    while clean.startswith("_orig_mod."):
        clean = clean[len("_orig_mod."):]

    parts = [part for part in clean.split(".") if part]
    variants: list[str] = []
    for start in range(len(parts)):
        suffix = ".".join(parts[start:])
        variants.append(suffix)

        aliases = (
            ("classifier.", "head."),
            ("classification_head.", "head."),
            ("cls_head.", "head."),
            ("fc.", "head."),
            ("head.fc.", "head."),
            ("head.classifier.", "head."),
        )
        for source, target in aliases:
            if suffix.startswith(source):
                variants.append(target + suffix[len(source):])

    # Keep order while removing duplicates.
    return list(dict.fromkeys(variants))


def map_checkpoint_state(
    model: nn.Module,
    raw_state: Mapping[str, torch.Tensor],
) -> tuple[dict[str, torch.Tensor], dict[str, Any]]:
    """Map arbitrary nested-wrapper names onto the exact model state keys."""

    expected = model.state_dict()
    mapped: dict[str, torch.Tensor] = {}
    source_for: dict[str, str] = {}
    conflicts: list[dict[str, str]] = []
    wrong_shapes: list[dict[str, Any]] = []

    for raw_key, tensor in raw_state.items():
        compatible: list[str] = []
        for variant in key_variants(raw_key):
            if variant not in expected:
                continue
            if tuple(tensor.shape) == tuple(expected[variant].shape):
                compatible.append(variant)
            else:
                wrong_shapes.append({
                    "raw_key": raw_key,
                    "target": variant,
                    "checkpoint_shape": list(tensor.shape),
                    "expected_shape": list(expected[variant].shape),
                })

        if not compatible:
            continue

        # The longest target is the most specific suffix match.
        target = max(compatible, key=lambda item: (item.count("."), len(item)))
        if target in mapped:
            conflicts.append({
                "target": target,
                "first": source_for[target],
                "second": raw_key,
            })
            continue
        mapped[target] = tensor
        source_for[target] = raw_key

    # Some custom wrappers use an indexed MLP/classifier name. Resolve only the
    # two-class head by semantic token + exact shape, and only if unambiguous.
    for target in ("head.weight", "head.bias"):
        if target in mapped or target not in expected:
            continue
        candidates: list[tuple[str, torch.Tensor]] = []
        for raw_key, tensor in raw_state.items():
            lower = raw_key.lower()
            semantic = any(
                token in lower
                for token in ("head", "classifier", "classification", "logits", ".fc")
            )
            if semantic and tuple(tensor.shape) == tuple(expected[target].shape):
                candidates.append((raw_key, tensor))
        if len(candidates) == 1:
            raw_key, tensor = candidates[0]
            mapped[target] = tensor
            source_for[target] = raw_key

    missing = sorted(set(expected) - set(mapped))
    used_raw = set(source_for.values())
    unexpected = sorted(set(raw_state) - used_raw)
    diagnostics = {
        "matching": len(mapped),
        "expected": len(expected),
        "missing": missing,
        "unexpected_count": len(unexpected),
        "unexpected_sample": unexpected[:50],
        "shape_mismatch": wrong_shapes[:50],
        "conflicts": conflicts[:50],
        "source_examples": dict(list(source_for.items())[:30]),
        "raw_key_sample": list(raw_state)[:50],
    }
    return mapped, diagnostics


def checkpoint_candidate_report(
    payload: Any,
) -> tuple[VideoMAEv2Giant, dict[str, torch.Tensor], dict[str, Any]]:
    """Select the actual model/EMA state and pooling variant by compatibility."""

    candidates = list(iter_tensor_mappings(payload))
    if not candidates:
        raise RuntimeError(
            "No tensor state dictionary was found inside best.pth. "
            f"Top-level payload type: {type(payload).__name__}"
        )

    evaluations: list[dict[str, Any]] = []
    best: tuple[
        tuple[int, int, int],
        VideoMAEv2Giant,
        dict[str, torch.Tensor],
        dict[str, Any],
    ] | None = None

    for mapping_path, raw_state in candidates:
        for use_mean_pooling in (False, True):
            with torch.device("meta"):
                model = VideoMAEv2Giant(
                    num_classes=2,
                    use_mean_pooling=use_mean_pooling,
                    initialize=False,
                )
            mapped, diagnostics = map_checkpoint_state(model, raw_state)
            score = (
                diagnostics["matching"],
                -len(diagnostics["missing"]),
                -len(diagnostics["shape_mismatch"]),
            )
            summary = {
                "mapping_path": mapping_path,
                "raw_tensor_count": len(raw_state),
                "use_mean_pooling": use_mean_pooling,
                "score": list(score),
                "matching": diagnostics["matching"],
                "expected": diagnostics["expected"],
                "missing_sample": diagnostics["missing"][:20],
                "raw_key_sample": list(raw_state)[:20],
            }
            evaluations.append(summary)

            if best is None or score > best[0]:
                full_report = {
                    "selected_mapping_path": mapping_path,
                    "selected_use_mean_pooling": use_mean_pooling,
                    "raw_tensor_count": len(raw_state),
                    **diagnostics,
                }
                best = (score, model, mapped, full_report)

    assert best is not None
    _, model, mapped, report = best
    report["all_candidates"] = sorted(
        evaluations,
        key=lambda item: tuple(item["score"]),
        reverse=True,
    )[:30]
    return model, mapped, report


def load_checkpoint_payload(path: Path) -> Any:
    """Load safely with mmap when supported, with explicit fallbacks."""

    attempts = (
        {"mmap": True, "weights_only": True},
        {"mmap": True, "weights_only": False},
        {"weights_only": True},
        {"weights_only": False},
    )
    errors: list[str] = []
    for options in attempts:
        try:
            print("torch.load options:", options)
            return torch.load(path, map_location="cpu", **options)
        except Exception as error:
            errors.append(f"{options}: {type(error).__name__}: {error}")
            print("Checkpoint load attempt failed:", errors[-1][:2000])
    raise RuntimeError(
        "All torch.load strategies failed.\n" + "\n".join(errors)
    )


def load_released_giant(path: Path) -> VideoMAEv2Giant:
    print("Opening released checkpoint:", path)
    payload = load_checkpoint_payload(path)
    model, state, report = checkpoint_candidate_report(payload)

    atomic_json(report, OUTPUT_DIR / "checkpoint_diagnostics.json")
    print(json.dumps({
        "selected_mapping_path": report["selected_mapping_path"],
        "selected_use_mean_pooling": report["selected_use_mean_pooling"],
        "matching": report["matching"],
        "expected": report["expected"],
        "raw_tensor_count": report["raw_tensor_count"],
        "missing_sample": report["missing"][:20],
        "raw_key_sample": report["raw_key_sample"][:10],
    }, indent=2))

    if report["missing"] or report["shape_mismatch"] or report["conflicts"]:
        raise RuntimeError(
            "The released checkpoint was found, but automatic structural mapping "
            "could not produce an exact state dictionary. "
            f"selected_path={report['selected_mapping_path']}, "
            f"mean_pooling={report['selected_use_mean_pooling']}, "
            f"matching={report['matching']}/{report['expected']}, "
            f"missing={report['missing'][:20]}, "
            f"shape_mismatch={report['shape_mismatch'][:5]}, "
            f"conflicts={report['conflicts'][:5]}. "
            "See checkpoint_diagnostics.json for raw key samples and all candidates."
        )

    incompatible = model.load_state_dict(state, strict=True, assign=True)
    if incompatible.missing_keys or incompatible.unexpected_keys:
        raise RuntimeError(f"Strict checkpoint assignment failed: {incompatible}")

    model = model.to(device=DEVICE, dtype=torch.float16).eval()
    model.checkpoint_mapping_path = report["selected_mapping_path"]
    model.checkpoint_use_mean_pooling = report["selected_use_mean_pooling"]

    del state, payload
    gc.collect()
    torch.cuda.empty_cache()
    return model


MODEL = load_released_giant(CHECKPOINT)
loaded_parameters = sum(parameter.numel() for parameter in MODEL.parameters())
if loaded_parameters != PARAMETER_COUNT:
    raise RuntimeError(
        f"Loaded parameter count mismatch: {loaded_parameters:,} vs {PARAMETER_COUNT:,}"
    )
print(
    "Strict checkpoint load passed:",
    f"{loaded_parameters:,}",
    "parameters; mapping=",
    MODEL.checkpoint_mapping_path,
    "; mean_pooling=",
    MODEL.checkpoint_use_mean_pooling,
)


Opening released checkpoint: /kaggle/working/nexar_v82_cache/checkpoint/best.pth
torch.load options: {'mmap': True, 'weights_only': True}
{
  "selected_mapping_path": "<root>.model",
  "selected_use_mean_pooling": false,
  "matching": 526,
  "expected": 526,
  "raw_tensor_count": 526,
  "missing_sample": [],
  "raw_key_sample": [
    "backbone.model.patch_embed.proj.weight",
    "backbone.model.patch_embed.proj.bias",
    "backbone.model.blocks.0.norm1.weight",
    "backbone.model.blocks.0.norm1.bias",
    "backbone.model.blocks.0.attn.q_bias",
    "backbone.model.blocks.0.attn.v_bias",
    "backbone.model.blocks.0.attn.qkv.weight",
    "backbone.model.blocks.0.attn.proj.weight",
    "backbone.model.blocks.0.attn.proj.bias",
    "backbone.model.blocks.0.norm2.weight"
  ]
}
Strict checkpoint load passed: 1,011,613,570 parameters; mapping= <root>.model ; mean_pooling= False


## 9. GPU Preflight واقعی با Input کامل 16×224×224


In [10]:
def collision_probability(logits: torch.Tensor) -> torch.Tensor:
    if logits.ndim != 2 or logits.shape[1] != 2:
        raise ValueError(f"Expected [B,2] logits, got {tuple(logits.shape)}")
    return torch.softmax(logits.float() / CFG.temperature, dim=1)[:, 1]


torch.cuda.reset_peak_memory_stats()
synthetic = torch.zeros(
    1, 3, CFG.num_frames, CFG.image_size, CFG.image_size,
    device=DEVICE,
    dtype=torch.float16,
)
with torch.inference_mode(), torch.autocast(
    device_type="cuda",
    dtype=AMP_DTYPE,
    enabled=AMP_ENABLED,
):
    synthetic_logits = MODEL(synthetic)
synthetic_score = collision_probability(synthetic_logits)
torch.cuda.synchronize()

assert synthetic_logits.shape == (1, 2)
assert torch.isfinite(synthetic_logits).all()
assert 0.0 <= float(synthetic_score.item()) <= 1.0
print(
    "Full giant forward PASS; peak GPU GiB:",
    round(torch.cuda.max_memory_allocated() / 2**30, 3),
)
del synthetic, synthetic_logits, synthetic_score
torch.cuda.empty_cache()


Full giant forward PASS; peak GPU GiB: 1.984


## 10. Inference دقیق Test، Resume، True IDs و Submission


In [11]:
TEST_MANIFEST = CACHE_DIR / (
    "test_v82_smoke.jsonl" if CFG.stage == "smoke" else "test_v82_released_giant.jsonl"
)
FAILURE_REPORT = OUTPUT_DIR / "test_failure_report.csv"
FAILURE_SUMMARY = OUTPUT_DIR / "test_failure_summary.json"


def predict_clip_batch(clips: Sequence[torch.Tensor]) -> tuple[list[float], list[list[float]]]:
    if not clips:
        return [], []
    tensor = torch.stack(clips).to(
        device=DEVICE,
        dtype=torch.float16,
        non_blocking=True,
    )
    logits: torch.Tensor | None = None
    probabilities: torch.Tensor | None = None
    try:
        with torch.inference_mode(), torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
            enabled=AMP_ENABLED,
        ):
            logits = MODEL(tensor)
        probabilities = collision_probability(logits)
        return probabilities.cpu().tolist(), logits.float().cpu().tolist()
    finally:
        del tensor, logits, probabilities


def predict_one_video(path: Path) -> dict[str, Any]:
    clips, metadata = final_visible_clips(path)
    try:
        scores, logits = predict_clip_batch(clips)
    finally:
        del clips
    if not scores:
        raise RuntimeError("No TTA clip score was produced.")
    return {
        "score": float(np.mean(scores)),
        "score_std": float(np.std(scores)),
        "tta_scores": scores,
        "tta_logits": logits,
        **metadata,
    }


def failure_signature(error: BaseException) -> str:
    message = str(error).strip().splitlines()[0] if str(error).strip() else ""
    message = re.sub(r"/tmp/[^\s,'\"]+", "/tmp/<file>", message)
    return f"{type(error).__name__}: {message}"[:500]


def process_test_example(
    example: Mapping[str, Any],
    index: int,
    identifier: str,
) -> dict[str, Any]:
    started = time.perf_counter()
    last_error: BaseException | None = None
    for attempt in range(1, CFG.test_retries + 1):
        stage = "materialize"
        try:
            with temporary_video(
                example["video"],
                identifier=identifier,
                hf_key=str(example.get("__hf_key__", "")) or None,
            ) as path:
                stage = "probe/decode/model"
                prediction = predict_one_video(path)
            return {
                "video_id": identifier,
                "dataset_index": index,
                "status": "ok",
                "attempt": attempt,
                "elapsed_seconds": time.perf_counter() - started,
                **prediction,
            }
        except Exception as error:
            last_error = error
            if isinstance(error, torch.cuda.OutOfMemoryError) or "out of memory" in str(error).lower():
                gc.collect()
                torch.cuda.empty_cache()
            if attempt < CFG.test_retries:
                time.sleep(min(2 ** (attempt - 1), 4))

    assert last_error is not None
    return {
        "video_id": identifier,
        "dataset_index": index,
        "status": "failed",
        "attempt": CFG.test_retries,
        "failure_stage": stage,
        "failure_signature": failure_signature(last_error),
        "elapsed_seconds": time.perf_counter() - started,
        "error": repr(last_error),
        "traceback": "".join(
            traceback.format_exception(
                type(last_error), last_error, last_error.__traceback__
            )
        )[-12000:],
        "video_field_type": type(example.get("video")).__name__,
        "video_path_hint": video_path_hint(example.get("video")),
        "hf_key": str(example.get("__hf_key__", "")),
    }


def save_failure_diagnostics(latest: pd.DataFrame) -> pd.DataFrame:
    failures = latest.loc[latest.status.eq("failed")].copy()
    if failures.empty:
        FAILURE_REPORT.unlink(missing_ok=True)
        FAILURE_SUMMARY.unlink(missing_ok=True)
        return failures

    failures.to_csv(FAILURE_REPORT, index=False)
    signature_counts = (
        failures["failure_signature"].fillna("unknown")
        .value_counts(dropna=False)
        .rename_axis("signature")
        .reset_index(name="count")
    )
    payload = {
        "failed_videos": int(len(failures)),
        "successful_videos": int((latest.status == "ok").sum()),
        "failure_report": str(FAILURE_REPORT),
        "top_failure_signatures": signature_counts.head(20).to_dict("records"),
        "sample_failures": failures[
            [
                column for column in (
                    "video_id", "dataset_index", "failure_stage",
                    "failure_signature", "video_path_hint", "hf_key", "error",
                ) if column in failures
            ]
        ].head(20).to_dict("records"),
    }
    atomic_json(payload, FAILURE_SUMMARY)
    display(signature_counts.head(20))
    display(
        failures[
            [column for column in (
                "video_id", "failure_stage", "failure_signature",
                "video_path_hint", "hf_key",
            ) if column in failures]
        ].head(20)
    )
    print("Detailed failure report:", FAILURE_REPORT)
    print("Failure summary:", FAILURE_SUMMARY)
    return failures


def infer_test(maximum: int | None) -> pd.DataFrame:
    complete = completed_ids(TEST_MANIFEST) if CFG.resume else set()
    raw = load_raw_split(CFG.test_split)
    total = maximum
    if total is None and "sample_submission.csv" in ARTIFACTS:
        total = len(pd.read_csv(ARTIFACTS["sample_submission.csv"]))

    successes_this_run = 0
    early_signatures: list[str] = []

    for index, example in tqdm(enumerate(raw), total=total, desc="Released giant Test"):
        if maximum is not None and index >= maximum:
            break
        identifier = example_id(example, index)
        if identifier in complete:
            continue

        record = process_test_example(example, index, identifier)
        append_jsonl(record, TEST_MANIFEST)

        if record["status"] == "ok":
            complete.add(identifier)
            successes_this_run += 1
            early_signatures.clear()
        else:
            early_signatures.append(str(record.get("failure_signature", "unknown")))
            # A repeated error before any successful example is systemic. Abort early
            # instead of wasting hours and producing 1,344 identical failures.
            if (
                successes_this_run == 0
                and len(early_signatures) >= CFG.systemic_failure_limit
                and len(set(early_signatures[-CFG.systemic_failure_limit:])) == 1
            ):
                latest = read_jsonl_recover(TEST_MANIFEST).drop_duplicates(
                    "video_id", keep="last"
                )
                save_failure_diagnostics(latest)
                raise RuntimeError(
                    "Systemic Test failure detected on the first "
                    f"{CFG.systemic_failure_limit} videos: "
                    f"{early_signatures[-1]}. See {FAILURE_REPORT}."
                )

    frame = read_jsonl_recover(TEST_MANIFEST)
    if frame.empty:
        raise RuntimeError("No Test inference records were produced.")
    latest = frame.drop_duplicates("video_id", keep="last")
    failures = save_failure_diagnostics(latest)
    if not failures.empty:
        top = failures["failure_signature"].fillna("unknown").value_counts().head(3)
        raise RuntimeError(
            f"{len(failures)} Test videos still fail after {CFG.test_retries} attempts. "
            f"Top causes: {top.to_dict()}. Details: {FAILURE_REPORT}"
        )
    return latest.loc[latest.status.eq("ok")].sort_values("dataset_index").reset_index(drop=True)


def prediction_table(frame: pd.DataFrame) -> pd.DataFrame:
    table = frame[["video_id", "score"]].copy()
    table["id"] = table.video_id.map(canonical_video_id)
    if table.id.isna().any() or table.id.duplicated().any():
        raise RuntimeError("Prediction IDs are missing or duplicated.")
    if not np.isfinite(table.score).all() or not table.score.between(0, 1).all():
        raise RuntimeError("Prediction scores must be finite and in [0,1].")
    return table[["id", "score"]]


def create_submission(predictions: pd.DataFrame, path: Path) -> Path:
    sample = pd.read_csv(
        ARTIFACTS["sample_submission.csv"],
        dtype={"id": str},
    )[["id", "score"]]
    sample["canonical_id"] = sample.id.map(canonical_video_id)
    table = prediction_table(predictions).rename(columns={"id": "canonical_id"})
    merged = sample[["id", "canonical_id"]].merge(
        table,
        on="canonical_id",
        how="left",
        validate="one_to_one",
    )
    if merged.score.isna().any():
        missing = merged.loc[merged.score.isna(), "id"].tolist()
        raise RuntimeError(
            f"Missing {len(missing)} Test IDs: {missing[:20]}. "
            "The Test inference must finish successfully before Submission creation."
        )
    submission = merged[["id", "score"]]
    if len(submission) != len(sample) or not submission.id.equals(sample.id):
        raise AssertionError("Submission does not preserve official row order.")
    submission.to_csv(path, index=False)
    return path


TEST_RESULTS: pd.DataFrame | None = None
SUBMISSION_PATH: Path | None = None

if CFG.stage in {"smoke", "reproduce"}:
    TEST_RESULTS = infer_test(CFG.max_test_videos)
    TEST_RESULTS.to_csv(PREDICTION_DIR / "released_giant_test_predictions.csv", index=False)

    if CFG.max_test_videos is None:
        SUBMISSION_PATH = create_submission(
            TEST_RESULTS,
            OUTPUT_DIR / "submission.csv",
        )
        print("Submission:", SUBMISSION_PATH)
    else:
        partial = prediction_table(TEST_RESULTS)
        SUBMISSION_PATH = OUTPUT_DIR / "smoke_scores.csv"
        partial.to_csv(SUBMISSION_PATH, index=False)
        print("Smoke output:", SUBMISSION_PATH)

    display(pd.read_csv(SUBMISSION_PATH, dtype={"id": str}).head())
else:
    print("Training stage selected; Test inference is skipped until the head is locked.")


Disabled HF video backends: {'TORCHCODEC_AVAILABLE': False, 'TORCHVISION_AVAILABLE': False}


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/1502 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1348 [00:00<?, ?it/s]

Released giant Test:   0%|          | 0/1344 [00:00<?, ?it/s]

Submission: /kaggle/working/outputs/videomaev2_giant_v82_reproduce/submission.csv


,id,score
0,00204,0.182280
1,00030,0.882933
2,00146,0.852506
3,00020,0.949295
4,00511,0.786787


## 11. ارزیابی رسمی mAP و Accuracy ثانویه


In [16]:
def evaluate_official(submission_path: Path) -> dict[str, Any]:
    if "solution.csv" not in ARTIFACTS:
        raise FileNotFoundError("Official solution.csv is unavailable.")
    prediction = pd.read_csv(submission_path, dtype={"id": str})
    solution = pd.read_csv(ARTIFACTS["solution.csv"], dtype={"id": str})
    required = {"id", "target", "group", "Usage"}
    if not required.issubset(solution.columns):
        raise RuntimeError(
            f"solution.csv missing columns: {sorted(required - set(solution.columns))}"
        )

    merged = solution.merge(prediction, on="id", validate="one_to_one")
    if len(merged) != len(solution):
        raise RuntimeError("Submission coverage does not match solution.csv.")

    report: dict[str, Any] = {
        "primary_metric": "mean Average Precision across official groups",
        "temperature": CFG.temperature,
        "accuracy_note": (
            "Accuracy is diagnostic only and uses fixed threshold 0.5; "
            "it is not the Kaggle ranking metric."
        ),
    }
    for usage in ("Public", "Private"):
        subset = merged.loc[merged.Usage.eq(usage)]
        group_ap = [
            average_precision_numpy(group.target.astype(int), group.score.astype(float))
            for _, group in subset.groupby("group", sort=True)
        ]
        report[usage.lower()] = {
            "official_map": float(np.nanmean(group_ap)),
            "groups": int(len(group_ap)),
            "n": int(len(subset)),
            "accuracy_at_0_5": binary_metrics(
                subset.target.astype(int),
                subset.score.astype(float),
                0.5,
            )["accuracy"],
        }
    report["overall"] = binary_metrics(
        merged.target.astype(int),
        merged.score.astype(float),
        0.5,
    )
    return report


OFFICIAL_METRICS: dict[str, Any] = {}
if (
    CFG.stage == "reproduce"
    and SUBMISSION_PATH is not None
    and CFG.max_test_videos is None
    and CFG.evaluate_official_solution
    and "solution.csv" in ARTIFACTS
):
    OFFICIAL_METRICS = evaluate_official(SUBMISSION_PATH)
    atomic_json(OFFICIAL_METRICS, OUTPUT_DIR / "official_metrics.json")
    print(json.dumps(OFFICIAL_METRICS, indent=2))
elif CFG.stage in {"smoke", "reproduce"}:
    print("Official full-Test evaluation skipped.")


{
  "primary_metric": "mean Average Precision across official groups",
  "temperature": 2.0,
  "accuracy_note": "Accuracy is diagnostic only and uses fixed threshold 0.5; it is not the Kaggle ranking metric.",
  "public": {
    "official_map": 0.8189942290608901,
    "groups": 3,
    "n": 667,
    "accuracy_at_0_5": 0.7256371814092953
  },
  "private": {
    "official_map": 0.8093667583147659,
    "groups": 3,
    "n": 677,
    "accuracy_at_0_5": 0.7311669128508124
  },
  "overall": {
    "accuracy": 0.7284226190476191,
    "balanced_accuracy": 0.7284226190476191,
    "precision": 0.7247437774524158,
    "recall": 0.7366071428571429,
    "specificity": 0.7202380952380952,
    "f1": 0.7306273062730627,
    "average_precision": 0.8216629634632804,
    "threshold": 0.5,
    "tp": 495,
    "tn": 484,
    "fp": 188,
    "fn": 177
  }
}


## 12. Pipeline حرفه‌ای Train/Fine‑tune قابل اجرا روی P100

این بخش برای بازآموزی کامل Giant نیست. مسیر قابل‌اجرا:

1. Split در سطح ویدئو؛ هیچ پنجره‌ای از یک ویدئو میان Train و Validation پخش نمی‌شود.
2. Positive windows فقط پیش از Event و در محدوده ۱٫۵ ثانیه آخر.
3. Negative windows از ویدئوهای منفی و بخش‌های زودتر ویدئوهای مثبت.
4. Random undersampling منفی برای تعادل.
5. استخراج Featureهای ۱۴۰۸بعدی با Backbone منتشرشده و ذخیره FP16.
6. آموزش Head با OOF پنج‌Fold.
7. انتخاب Threshold فقط روی OOF، نه Test.


In [17]:
def finite_float(value: Any) -> float | None:
    if value is None:
        return None
    try:
        result = float(value)
    except (TypeError, ValueError):
        return None
    return result if math.isfinite(result) else None


def deterministic_rng(video_id: str) -> np.random.Generator:
    digest = hashlib.sha256(f"{CFG.seed}:{video_id}".encode()).digest()
    seed = int.from_bytes(digest[:8], "little") % (2**32)
    return np.random.default_rng(seed)


def evenly_choose(values: np.ndarray, maximum: int) -> np.ndarray:
    if len(values) <= maximum:
        return values
    positions = np.linspace(0, len(values) - 1, maximum)
    return values[np.rint(positions).astype(int)]


def training_window_specs(
    video_id: str,
    metadata: Mapping[str, float | int],
    event_time: float | None,
) -> list[dict[str, Any]]:
    fps = float(metadata["fps"])
    frame_count = int(metadata["frame_count"])
    duration = float(metadata["duration"])
    min_end = (CFG.num_frames - 1) * CFG.frame_interval
    all_ends = np.arange(
        min_end,
        frame_count,
        CFG.training_window_stride,
        dtype=np.int64,
    )
    rng = deterministic_rng(video_id)

    if event_time is not None:
        event_end = causal_end_frame(event_time, fps, frame_count)
        positive_start_time = max(0.0, event_time - CFG.positive_horizon_seconds)
        positive_start = int(math.ceil(positive_start_time * fps))
        positive = all_ends[
            (all_ends >= positive_start)
            & (all_ends <= event_end)
        ]
        negative = all_ends[all_ends < positive_start]
        positive = evenly_choose(positive, CFG.positive_windows_per_video)
        if len(negative) > CFG.negative_windows_per_video:
            negative = np.sort(
                rng.choice(
                    negative,
                    size=CFG.negative_windows_per_video,
                    replace=False,
                )
            )
    else:
        positive = np.asarray([], dtype=np.int64)
        negative = all_ends
        if len(negative) > CFG.negative_windows_per_video:
            negative = np.sort(
                rng.choice(
                    negative,
                    size=CFG.negative_windows_per_video,
                    replace=False,
                )
            )

    specs = [
        {"end_frame": int(end), "label": 1}
        for end in positive
    ]
    specs.extend(
        {"end_frame": int(end), "label": 0}
        for end in negative
    )
    return specs


def extract_feature_batch(clips: Sequence[torch.Tensor]) -> np.ndarray:
    tensor = torch.stack(clips).to(
        device=DEVICE,
        dtype=torch.float16,
        non_blocking=True,
    )
    try:
        with torch.inference_mode(), torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
            enabled=AMP_ENABLED,
        ):
            features = MODEL.forward_features(tensor)
        return features.float().cpu().numpy().astype(np.float16)
    finally:
        del tensor


def cache_features_for_video(
    example: Mapping[str, Any],
    index: int,
) -> Path:
    identifier = example_id(example, index)
    destination = FEATURE_DIR / f"{identifier}.npz"
    if CFG.resume and destination.exists() and destination.stat().st_size > 1024:
        return destination

    with temporary_video(example["video"]) as path:
        metadata = probe_video(path)
        event_time = finite_float(example.get("time_of_event"))
        specs = training_window_specs(identifier, metadata, event_time)
        if not specs:
            raise RuntimeError(f"No training windows generated for {identifier}")

        features: list[np.ndarray] = []
        labels: list[int] = []
        end_frames: list[int] = []
        for start in range(0, len(specs), CFG.feature_batch_size):
            chunk = specs[start:start + CFG.feature_batch_size]
            clips: list[torch.Tensor] = []
            for spec in chunk:
                indices = clip_indices_from_end(
                    spec["end_frame"],
                    int(metadata["frame_count"]),
                )
                frames = decode_frame_indices(path, indices)
                clips.append(preprocess_clip(frames))
            features.append(extract_feature_batch(clips))
            labels.extend(int(spec["label"]) for spec in chunk)
            end_frames.extend(int(spec["end_frame"]) for spec in chunk)

    temporary = destination.with_suffix(".tmp.npz")
    np.savez_compressed(
        temporary,
        features=np.concatenate(features, axis=0),
        labels=np.asarray(labels, dtype=np.int8),
        end_frames=np.asarray(end_frames, dtype=np.int32),
        video_id=np.asarray([identifier]),
        video_label=np.asarray([int(event_time is not None)], dtype=np.int8),
    )
    os.replace(temporary, destination)
    return destination


def cache_training_features(maximum: int | None) -> list[Path]:
    outputs: list[Path] = []
    raw = load_raw_split(CFG.train_split)
    for index, example in tqdm(enumerate(raw), total=maximum, desc="Train feature cache"):
        if maximum is not None and index >= maximum:
            break
        try:
            outputs.append(cache_features_for_video(example, index))
        except Exception as error:
            identifier = example_id(example, index)
            append_jsonl(
                {"video_id": identifier, "status": "failed", "error": repr(error)},
                CACHE_DIR / "feature_failures.jsonl",
            )
            raise
    return outputs


FEATURE_FILES: list[Path] = []
if CFG.stage in {"cache_features", "train_head"}:
    FEATURE_FILES = cache_training_features(CFG.max_train_videos)
    print("Feature files:", len(FEATURE_FILES))


## 13. Five‑fold OOF Head Fine‑tuning، Early Stopping و Threshold Lock


In [18]:
class CollisionHead(nn.Module):
    def __init__(self, input_dim: int = 1408) -> None:
        super().__init__()
        self.network = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, CFG.head_hidden_dim),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(CFG.head_hidden_dim, 2),
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.network(features)


def load_feature_table(files: Sequence[Path]) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    all_features: list[np.ndarray] = []
    all_labels: list[np.ndarray] = []
    all_video_ids: list[np.ndarray] = []
    for path in files:
        with np.load(path) as payload:
            features = payload["features"]
            labels = payload["labels"]
            identifier = str(payload["video_id"][0])
        all_features.append(features)
        all_labels.append(labels)
        all_video_ids.append(np.repeat(identifier, len(labels)))
    return (
        np.concatenate(all_features).astype(np.float32),
        np.concatenate(all_labels).astype(np.int64),
        np.concatenate(all_video_ids).astype(str),
    )


def video_level_folds(video_ids: np.ndarray, labels: np.ndarray) -> dict[str, int]:
    frame = pd.DataFrame({"video_id": video_ids, "label": labels})
    video_labels = frame.groupby("video_id").label.max()
    assignment: dict[str, int] = {}
    for class_value, ids in video_labels.groupby(video_labels).groups.items():
        class_ids = sorted(str(item) for item in ids)
        rng = np.random.default_rng(CFG.seed + int(class_value))
        rng.shuffle(class_ids)
        for position, identifier in enumerate(class_ids):
            assignment[identifier] = position % CFG.folds
    return assignment


def train_one_head(
    features: np.ndarray,
    labels: np.ndarray,
    train_mask: np.ndarray,
    valid_mask: np.ndarray,
) -> tuple[CollisionHead, np.ndarray, dict[str, Any]]:
    model = CollisionHead(features.shape[1]).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CFG.head_lr,
        weight_decay=CFG.head_weight_decay,
    )
    criterion = nn.CrossEntropyLoss()

    x_train = torch.from_numpy(features[train_mask])
    y_train = torch.from_numpy(labels[train_mask])
    x_valid = torch.from_numpy(features[valid_mask]).to(DEVICE)
    y_valid = labels[valid_mask]

    best_ap = -1.0
    best_state: dict[str, torch.Tensor] | None = None
    best_epoch = -1
    stale = 0
    batch_size = 256

    for epoch in range(CFG.head_epochs):
        model.train()
        order = torch.randperm(len(x_train))
        for start in range(0, len(order), batch_size):
            indices = order[start:start + batch_size]
            batch_x = x_train[indices].to(DEVICE)
            batch_y = y_train[indices].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        with torch.inference_mode():
            valid_logits = model(x_valid)
            valid_scores = torch.softmax(valid_logits, dim=1)[:, 1].cpu().numpy()
        ap = average_precision_numpy(y_valid, valid_scores)
        if ap > best_ap + 1e-6:
            best_ap = ap
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            stale = 0
        else:
            stale += 1
            if stale >= CFG.head_patience:
                break

    if best_state is None:
        raise RuntimeError("Head training produced no valid checkpoint.")
    model.load_state_dict(best_state)
    model.eval()
    with torch.inference_mode():
        valid_scores = torch.softmax(model(x_valid), dim=1)[:, 1].cpu().numpy()
    return model, valid_scores, {
        "best_epoch": best_epoch,
        "best_ap": float(best_ap),
    }


HEAD_REPORT: dict[str, Any] = {}
if CFG.stage == "train_head":
    files = sorted(FEATURE_DIR.glob("*.npz"))
    if not files:
        raise RuntimeError("No feature cache files are available.")
    features, labels, video_ids = load_feature_table(files)
    fold_map = video_level_folds(video_ids, labels)
    folds = np.asarray([fold_map[item] for item in video_ids], dtype=np.int64)

    oof = np.full(len(labels), np.nan, dtype=np.float64)
    fold_reports: list[dict[str, Any]] = []
    for fold in range(CFG.folds):
        train_mask = folds != fold
        valid_mask = folds == fold
        _, scores, report = train_one_head(
            features,
            labels,
            train_mask,
            valid_mask,
        )
        oof[valid_mask] = scores
        fold_reports.append({"fold": fold, **report})

    if not np.isfinite(oof).all():
        raise RuntimeError("OOF prediction coverage is incomplete.")

    threshold, threshold_metrics = select_accuracy_threshold(labels, oof)
    HEAD_REPORT = {
        "oof_average_precision": average_precision_numpy(labels, oof),
        "oof_threshold": threshold,
        "oof_metrics": threshold_metrics,
        "folds": fold_reports,
        "samples": int(len(labels)),
        "videos": int(len(set(video_ids))),
        "warning": (
            "This is a P100-feasible head fine-tune. It is not a full "
            "1.03B-parameter AdamW fine-tune."
        ),
    }
    atomic_json(HEAD_REPORT, OUTPUT_DIR / "head_oof_report.json")
    pd.DataFrame({
        "video_id": video_ids,
        "label": labels,
        "fold": folds,
        "oof_score": oof,
    }).to_csv(PREDICTION_DIR / "head_oof_predictions.csv", index=False)
    print(json.dumps(HEAD_REPORT, indent=2))


## 14. Acceptance Tests، گزارش نهایی و مسیر خروجی‌ها


In [19]:
def checkpoint_mapper_self_test() -> bool:
    """Exercise the exact failure mode: nested wrappers and renamed classifier."""

    for use_mean_pooling in (False, True):
        with torch.device("meta"):
            test_model = VideoMAEv2Giant(
                num_classes=2,
                use_mean_pooling=use_mean_pooling,
                initialize=False,
            )
        expected = test_model.state_dict()
        wrapped: dict[str, torch.Tensor] = {}
        for key, tensor in expected.items():
            if key.startswith("head."):
                raw_key = "module.model.classifier." + key.split(".", 1)[1]
            else:
                raw_key = "module.model.model." + key
            wrapped[raw_key] = tensor
        mapped, diagnostics = map_checkpoint_state(test_model, wrapped)
        if diagnostics["missing"] or len(mapped) != len(expected):
            return False
    return True


def run_acceptance_tests() -> dict[str, bool]:
    tests = {
        "cuda_available": torch.cuda.is_available(),
        "giant_parameter_count": 1_000_000_000 <= PARAMETER_COUNT <= 1_080_000_000,
        "temperature_is_two": math.isclose(CFG.temperature, 2.0),
        "sixteen_frames": CFG.num_frames == 16,
        "window_stride_two": CFG.training_window_stride == 2,
        "positive_horizon_1_5": math.isclose(CFG.positive_horizon_seconds, 1.5),
        "checkpoint_size_gt_10_gib": CHECKPOINT.stat().st_size > 10 * 2**30,
        "checkpoint_model_loaded": loaded_parameters == PARAMETER_COUNT,
        "checkpoint_mapper_nested_wrapper_test": checkpoint_mapper_self_test(),
        "no_timm_import": "timm" not in globals(),
        "no_torchvision_import": "torchvision" not in globals(),
        "no_torchcodec_import": "torchcodec" not in globals(),
        "future_frame_exclusion": bool(
            np.all(
                clip_indices_from_end(
                    causal_end_frame(1.0, 30.0, 1200),
                    1200,
                ) / 30.0 < 1.0
            )
        ),
    }

    if CFG.stage == "reproduce" and CFG.max_test_videos is None:
        tests.update({
            "submission_exists": SUBMISSION_PATH is not None and SUBMISSION_PATH.exists(),
            "full_test_predictions": TEST_RESULTS is not None and len(TEST_RESULTS) > 100,
        })

    failed = [name for name, passed in tests.items() if not passed]
    if failed:
        raise AssertionError(f"Acceptance tests failed: {failed}")
    return tests


ACCEPTANCE_TESTS = run_acceptance_tests()
FINAL_REPORT = {
    "method": "Actual released Nexar VideoMAEv2-Giant solution",
    "checkpoint_repo": CFG.checkpoint_repo,
    "checkpoint_sha256": CFG.checkpoint_sha256,
    "parameter_count": PARAMETER_COUNT,
    "stage": CFG.stage,
    "public_claim": (
        "The released model card reports Public LB mAP 0.886. "
        "This notebook does not relabel that score as accuracy."
    ),
    "official_metrics_from_this_run": OFFICIAL_METRICS,
    "head_finetune_report": HEAD_REPORT,
    "submission": str(SUBMISSION_PATH) if SUBMISSION_PATH else None,
    "acceptance_tests": ACCEPTANCE_TESTS,
    "limitations": [
        "Exact leaderboard reproduction depends on the released best.pth, "
        "the exact public preprocessing details, and successful full Test inference.",
        "The public model card does not expose every optimizer, augmentation, "
        "split and TTA detail used during the original competition.",
        "Full AdamW fine-tuning of 1.03B parameters is not presented as P100-feasible.",
        "Accuracy is secondary; the official ranking metric is group-wise mAP.",
    ],
}
atomic_json(FINAL_REPORT, OUTPUT_DIR / "final_report.json")
print(json.dumps(FINAL_REPORT, indent=2, default=str))
print("\nOutput directory:", OUTPUT_DIR)
if SUBMISSION_PATH is not None:
    print("Submission:", SUBMISSION_PATH)
print("Final report:", OUTPUT_DIR / "final_report.json")


{
  "method": "Actual released Nexar VideoMAEv2-Giant solution",
  "checkpoint_repo": "zhiyaowang/VideoMaev2-giant-nexar-solution",
  "checkpoint_sha256": "dbf68c391c3322fe23a502a5e80afb7b3184221c63d62b25983507435fb39af7",
  "parameter_count": 1011613570,
  "stage": "reproduce",
  "public_claim": "The released model card reports Public LB mAP 0.886. This notebook does not relabel that score as accuracy.",
  "official_metrics_from_this_run": {
    "primary_metric": "mean Average Precision across official groups",
    "temperature": 2.0,
    "accuracy_note": "Accuracy is diagnostic only and uses fixed threshold 0.5; it is not the Kaggle ranking metric.",
    "public": {
      "official_map": 0.8189942290608901,
      "groups": 3,
      "n": 667,
      "accuracy_at_0_5": 0.7256371814092953
    },
    "private": {
      "official_map": 0.8093667583147659,
      "groups": 3,
      "n": 677,
      "accuracy_at_0_5": 0.7311669128508124
    },
    "overall": {
      "accuracy": 0.7284226190476